In [1]:
!nvidia-smi

Sun Nov  5 19:51:31 2023       

+-----------------------------------------------------------------------------+

| NVIDIA-SMI 525.116.04   Driver Version: 525.116.04   CUDA Version: 12.0     |

|-------------------------------+----------------------+----------------------+

| GPU  Name        Persistence-M| Bus-Id        Disp.A | Volatile Uncorr. ECC |

| Fan  Temp  Perf  Pwr:Usage/Cap|         Memory-Usage | GPU-Util  Compute M. |

|                               |                      |               MIG M. |

|===============================+======================+======================|

|   0  NVIDIA RTX A4000    Off  | 00000000:00:05.0 Off |                  Off |

| 41%   63C    P2   139W / 140W |  11711MiB / 16376MiB |     96%      Default |

|                               |                      |                  N/A |

+-------------------------------+----------------------+----------------------+

                                                                            

## Imports

In [2]:
import os
import random
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold
import torch
import torch.nn as nn
from torch.utils.checkpoint import checkpoint
from torch.utils.data import Dataset,DataLoader


from sklearn import metrics
from torch.nn import functional as F
from tqdm import tqdm
from transformers import (
    AdamW,
    AutoConfig,
    AutoModel,
    AutoTokenizer,
    get_cosine_schedule_with_warmup,
    get_linear_schedule_with_warmup,
    DataCollatorWithPadding
)
import warnings
warnings.filterwarnings('ignore')

## Configs

In [3]:
cfg = {"model_name": "microsoft/deberta-v3-large",
    "max_len": 1024,
    "freeze" : False,
    # Train Configs
    "fold_num": 5,
    "val_fold": 0,
    "learning_rate": 7.0e-06,
    "min_lr": 1e-6,
    "T_max": 600,
    "valid_batch_size": 4,
    'train_batch_size' : 2,
 
    "epochs": 5, # Set to 1 because it is a demo
    "accumulation_steps": 6,
    "val_steps": 375,
    
    # GPU Optimize Settings
    "scheduler" : 'cosine',
    " warmup_epochs": 1,

    "gradient_checkpoint" : False,
    'tokenizer' : AutoTokenizer.from_pretrained("microsoft/deberta-v3-large"),
    
    # Path
    "input": "input/",
    "output": "output/"
}

## Utils

In [4]:
def set_seed(seed=42):
    '''Sets the seed of the entire notebook so results are the same every time we run.
    This is for REPRODUCIBILITY.'''
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    # When running on the CuDNN backend, two further options must be set
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    # Set a fixed value for the hash seed
    os.environ['PYTHONHASHSEED'] = str(seed)
    
set_seed(42)

In [5]:
def optimizer_scheduler(model,steps):
        param_optimizer = list(model.named_parameters())
        no_decay = ["bias", "LayerNorm.weight"]
        optimizer_parameters = [
            {
                "params": [p for n, p in param_optimizer if not any(nd in n for nd in no_decay) and p.requires_grad],
                "weight_decay": 0.003,
            },
            {
                "params": [p for n, p in param_optimizer if any(nd in n for nd in no_decay) and p.requires_grad],
                "weight_decay": 0.0,
            },
        ]
        opt = AdamW(optimizer_parameters, lr=cfg['learning_rate'])

        sch = get_cosine_schedule_with_warmup(
            opt,
            num_warmup_steps=0,
            num_training_steps=steps,#self.num_train_steps,
            last_epoch=-1,
        )
        return opt, sch

In [6]:
import matplotlib.pyplot as plt
def plot_history(history):

    plt.figure(figsize=(20,6))
#     fig, axes = plt.subplots(1, 2, sharey=True, figsize=(22, 6))
    
    
    for k in ["Train Loss", "Valid Loss"]:
        plt.plot(history[k])

    plt.title('Loss')
    plt.xlabel('epochs')
    plt.ylabel('loss')
    plt.legend(['train', 'valid'], loc='upper left')


    
    plt.show()

## Dataset

In [7]:
train = pd.read_csv('input/train.csv')

In [9]:
train['Response'] = train['Response'].fillna('')

In [10]:
rows = []
for i, row in train.iterrows():
    question = row.Question
    response = row.Response
    context = train[(train.Question==question)&(train.Response!=response)].Response.values
    context = ' [SEP] '.join(context)
    train.loc[i, 'context'] = context

In [11]:
train

,Question,Response,target,context
0,Explain the concept of coevolution.,Coevolution is a biological process that occur...,3,Coevolution refers to the process where two or...
1,Is it possible that recurring fever and chills...,"Yes, recurring fever and chills can be a sympt...",4,"No, recurring fever and chills alone cannot co..."
2,Evaluate the expression 3!,The expression 3! represents the factorial of ...,1,The value of 3! (3 factorial) is equal to 3 × ...
3,What are the roles of different types of RNA i...,1. Messenger RNA (mRNA): mRNA carries genetic ...,3,There are several types of RNA molecules that ...
4,What is the role of gene flow in population ge...,Gene flow refers to the movement of individual...,3,Gene flow plays a crucial role in population g...
...,...,...,...,...
3971,You will always find me in the past. I can be ...,Please select one of the options from the tabl...,0,You are a memory. Memories are a recollection ...
3972,To give me to someone i don’t belong to is cow...,________________________________. (20 points),0,"You are ""a fight"". A fight can be a game, but ..."
3973,Could severe abdominal pain and vomiting be in...,Abdominal pain and vomiting can indeed be symp...,1,"Yes, severe abdominal pain and vomiting can be..."
3974,How can I increase my productivity at work or ...,1. Prioritize tasks: Make a to-do list with th...,1,There are several strategies you can use to in...


In [12]:
train['Question'].nunique()

568

In [13]:
TARGETS = ["target"]

In [14]:
class Dataset(Dataset) :
    def __init__(self,df,is_train = True, use_aug=False) :
        self.use_aug = use_aug
        self.df = df
        self.tokenizer = cfg['tokenizer']
        self.sep_token = self.tokenizer.sep_token
        self.text = (self.df['Question'] + self.sep_token + '[R_STRAT]' + self.df['Response']+ '[R_END]'+ self.sep_token + self.df['context']).values
        if is_train==True :
            self.targets = df['target'].values
        self.is_train = is_train
    def __len__(self):
        return len(self.df)
    def __getitem__(self, index):
        
        text = self.text[index]
        
        inputs = self.tokenizer.encode_plus(
            text,
            truncation=True,
            add_special_tokens=True,
            max_length=cfg["max_len"]
        )
        feature_dict = {
            'input_ids': inputs['input_ids'],
            'attention_mask': inputs['attention_mask'],
        }
        if self.is_train==True :
            feature_dict['target'] =  self.targets[index]
            
        custom_mask = inputs['attention_mask'].copy()#[0]*len(inputs['attention_mask'])
        stop_idx = 0
        for idx, token in enumerate(inputs['input_ids']):
                if token == cfg["tokenizer"].convert_tokens_to_ids('[R_END]'):
                    stop_idx = idx + 1
                    break
        for idx in range(stop_idx, len(inputs['attention_mask'])):
            custom_mask[idx] = 0
        feature_dict['R_mask'] = custom_mask

        return feature_dict

In [15]:
cfg["tokenizer"].add_tokens(['[R_STRAT]', '[R_END]'], special_tokens=True)

2

## Model

In [16]:
def odd_layer_freeze(module):
    for i in range(1,24,2):
        for n,p in module.encoder.layer[i].named_parameters():
            p.requires_grad = False
            
def even_layer_freeze(module):
    for i in range(0,24,2):
        for n,p in module.encoder.layer[i].named_parameters():
            p.requires_grad = False
            
def top_half_layer_freeze(module):
    for i in range(0,13,1):
        for n,p in module.encoder.layer[i].named_parameters():
            p.requires_grad = False

def bottom_half_layer_freeze(module):
    for i in range(13,14,1):
        for n,p in module.encoder.layer[i].named_parameters():
            p.requires_grad = False
    

In [17]:
class MeanPooling(nn.Module):
    def __init__(self):
        super(MeanPooling, self).__init__()
        
    def forward(self, last_hidden_state, attention_mask):
        input_mask_expanded = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
        sum_embeddings = torch.sum(last_hidden_state * input_mask_expanded, 1)
        sum_mask = input_mask_expanded.sum(1)
        sum_mask = torch.clamp(sum_mask, min=1e-9)
        mean_embeddings = sum_embeddings / sum_mask
        return mean_embeddings
    
class GeMText(nn.Module):
    def __init__(self, dim=1, p=3, eps=1e-6):
        super(GeMText, self).__init__()
        self.dim = dim
        self.p = Parameter(torch.ones(1) * p)
        self.eps = eps
        self.feat_mult = 1

    def forward(self, x, attention_mask):
        attention_mask_expanded = attention_mask.unsqueeze(-1).expand(x.shape)
        x = (x.clamp(min=self.eps) * attention_mask_expanded).pow(self.p).sum(self.dim)
        ret = x / attention_mask_expanded.sum(self.dim).clip(min=self.eps)
        ret = ret.pow(1 / self.p)
        return ret
    
class Model(nn.Module):
    def __init__(self):
        super(Model, self).__init__()
        self.drop = nn.Dropout(p=cfg["dropout"])
        
        self.config = AutoConfig.from_pretrained(cfg["model_name"])
        #self.config.hidden_dropout = 0.
        self.config.hidden_dropout_prob = 0.007
        #self.config.attention_dropout = 0.
        self.config.attention_probs_dropout_prob = 0.008

        self.model = AutoModel.from_pretrained(cfg["model_name"], config=self.config)
        #odd_layer_freeze(self.model)
        if cfg["gradient_checkpoint"]:
            print('Enabling Grad Checkpointing')
            self.model.gradient_checkpointing_enable()  
        if cfg["freeze"]:
            print('freezing params')
            for parameter in self.model.parameters():
                parameter.requires_grad = False
        self.pool = GeMText()
        self.fc = nn.Sequential(
            nn.Linear(self.config.hidden_size, 7),
        )
        
    def forward(self, ids, mask, rhead):        
        out = self.model(input_ids=ids,attention_mask=mask,
                         output_hidden_states=False)
        out = self.pool(out.last_hidden_state, rhead)
        out = self.drop(out)
        out = self.fc(out)
        return out

In [18]:
class Collate:
    def __init__(self, tokenizer, max_len=512):
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __call__(self, batch):
        output = dict()
        output["input_ids"] = [sample["input_ids"] for sample in batch]
        output["attention_mask"] = [sample["attention_mask"] for sample in batch]
        output["R_mask"] = [sample["R_mask"] for sample in batch]
        output["target"] = [sample["target"] for sample in batch]

        # calculate max token length of this batch
        batch_max = max([len(ids) for ids in output["input_ids"]]) 

        batch_max = min(batch_max, self.max_len)
        
        output["input_ids"] = [s[:batch_max] for s in output["input_ids"]]
        output["attention_mask"] = [s[:batch_max] for s in output["attention_mask"]] 
        output["R_mask"] = [s[:batch_max] for s in output["R_mask"]] 
        #output["target"] = [s[:batch_max] for s in output["target"]]


        output["input_ids"] = [s + (batch_max - len(s)) * [self.tokenizer.pad_token_id] for s in output["input_ids"]]
        output["attention_mask"] = [s + (batch_max - len(s)) * [0] for s in output["attention_mask"]]
        output["R_mask"] = [s + (batch_max - len(s)) * [0] for s in output["R_mask"]]
        #output["target"] = [s + (batch_max - len(s)) * [0] for s in output["target"]]
        

        # convert to tensors
        output["input_ids"] = torch.tensor(output["input_ids"], dtype=torch.long)
        output["attention_mask"] = torch.tensor(output["attention_mask"], dtype=torch.long)
        output["target"] = torch.tensor(output["target"], dtype=torch.long)
        output["R_mask"] = torch.tensor(output["R_mask"], dtype=torch.long)

        return output

## Engine

In [19]:
class AverageMeter(object):
    """Computes and stores the average and current value"""
    def __init__(self):
        self.reset()

    def reset(self):
        self.val = 0
        self.avg = 0
        self.sum = 0
        self.count = 0

    def update(self, val, n=1):
        self.val = val
        self.sum += val * n
        self.count += n
        self.avg = self.sum / self.count

In [20]:
def train_one_epoch(model, optimizer, scheduler, dataloader, device):
    model.train()

    total = 0
    running_loss = 0.0
    losses = AverageMeter()
    scaler = torch.cuda.amp.GradScaler(enabled=cfg["apex"])
    lr = []
    bar = tqdm(dataloader, total=len(dataloader))
    steps = len(dataloader)
    
    all_preds = np.array([])
    all_groud_truth = np.array([])
    
    for step, data in enumerate(bar):
        ids = data["input_ids"].to(device, dtype=torch.long)
        mask = data["attention_mask"].to(device, dtype=torch.long)
        targets = data["target"].to(device, dtype=torch.long)
        r_mask = data["R_mask"].to(device, dtype=torch.long)

        batch_size = ids.size(0)
        with torch.cuda.amp.autocast(enabled=cfg["apex"]):
            outputs = model(ids, mask, r_mask)
            loss = criterion(outputs, targets)
        loss = loss / cfg['n_accumulate']
        
        losses.update(loss.item(), batch_size)
        scaler.scale(loss).backward()
        
        if (step + 1) % cfg['n_accumulate'] == 0 or step == steps:
            #scaler.unscale_(optimizer)
            #grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), cfg["grad_norm"])
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
            if scheduler:
                scheduler.step()


        epoch_loss = losses.avg
        #acc = correct / total

        bar.set_postfix(
            Loss=epoch_loss, LR=optimizer.param_groups[0]['lr'])
        
        lr.append(optimizer.param_groups[0]['lr'])

    return losses.avg
from sklearn.metrics import roc_auc_score
@torch.no_grad()
def evaluate(model, dataloader, device):
    model.eval()

    total = 0
    losses = AverageMeter()
    correct = 0
    preds = []
    y_test = []
    for data in dataloader:
        ids = data["input_ids"].to(device, dtype=torch.long)
        mask = data["attention_mask"].to(device, dtype=torch.long)
        targets = data["target"].to(device, dtype=torch.long)
        r_mask = data["R_mask"].to(device, dtype=torch.long)

        batch_size = ids.size(0)

        outputs = model(ids, mask, r_mask)

        loss = criterion(outputs, targets)

        losses.update(loss.item(), batch_size)
        preds.append(outputs.detach().cpu().numpy())
        y_test.append(targets.detach().cpu().numpy())
        
    

    epoch_loss = losses
    preds = np.concatenate(preds)
    y_test = np.concatenate(y_test)

    return losses.avg, preds, y_test

In [21]:
!mkdir -p models/exp3

In [22]:
def start_training(model, optimizer, scheduler, device, num_epochs,train_loader,valid_loader,fold=0):
    start = time.time()
    best_score = np.inf
    history = {"Train Loss": [], "Valid Loss": [],"LR": []}
    

    for epoch in range(1, num_epochs + 1):
        print("Epoch: ", epoch)
        train_epoch_loss = train_one_epoch(
            model, optimizer, scheduler, dataloader=train_loader, device=cfg["device"]
        )

        val_epoch_loss, preds, y_test = evaluate(
            model, valid_loader, device=cfg["device"]
        )
        #score, scores = get_score(y_test, preds)
        #print(f'Epoch {epoch} : ',score, scores )
        score = val_epoch_loss

        history["Train Loss"].append(train_epoch_loss)
        history["Valid Loss"].append(val_epoch_loss)

        # deep copy the model
        if score <= best_score:
            print(
                f"Score Improved ({best_score} ---> {score})"
            )
            with open('log3.txt', 'a') as f:
                f.write(f'score {score} \n')
            best_score = score
            PATH = f"models/exp3/fold_{fold}.bin"
            torch.save(model.state_dict(), PATH)
            
            # Save a model file from the current directory
            print(f"Model Saved")
            best_y = preds

        print()

    end = time.time()
    time_elapsed = end - start
    print(
        "Training complete in {:.0f}h {:.0f}m {:.0f}s".format(
            time_elapsed // 3600,
            (time_elapsed % 3600) // 60,
            (time_elapsed % 3600) % 60,
        )
    )
    print(
        "Best Score: {:.4f}".format(
            best_score
        )
    )


    return history, best_y

In [24]:
from sklearn.model_selection import GroupKFold
folds = GroupKFold(n_splits=5)
train['fold'] = -1
for i,(train_index, test_index) in enumerate(folds.split(train,train['target'], groups=train['Question'])): 
    train.loc[test_index,'fold'] = i

In [25]:
cfg["device"] = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [26]:
cfg['n_accumulate'] = 4#5
cfg['dropout'] = 0.2
cfg['apex'] = True
cfg["grad_norm"] = 20
cfg["gradient_checkpoint"] = False

In [27]:
from torch.optim import lr_scheduler
from sklearn.metrics import mean_squared_error
from torch.nn import Parameter
import time
criterion = nn.CrossEntropyLoss()#RMSELoss()
def run_folds() :
    
    for fold in range(5) :

        print(f'--------------------------------Training Fold {fold+1}/5---------------------------------')
        with open('log3.txt', 'a') as f:
                f.write(f'fold {fold+1}/5 \n')
        train_ = train[train.fold!=fold].reset_index()
        valid_ = train[train.fold==fold].reset_index()
        
        print(f'train shape : {len(train_)}')
        print(f'valid shape : {len(valid_)}')
        
        train_dataset = Dataset(
                                    train_, True, True
                            )
        valid_dataset = Dataset(
                                    valid_,True
        )
        collate_fn = Collate(tokenizer=cfg['tokenizer'])#DataCollatorWithPadding(tokenizer=cfg['tokenizer'])
        train_loader = DataLoader(
                train_dataset,
                batch_size=cfg["train_batch_size"],
                collate_fn=collate_fn,
                num_workers=4,
                shuffle=True,
                pin_memory=True,
                drop_last=True
                    )
        valid_loader = DataLoader(
            valid_dataset,
            batch_size=cfg["valid_batch_size"],
            collate_fn=collate_fn,
            num_workers=4,
            shuffle=False,
            pin_memory=True,
        )
        
        model = Model()
        model.to(cfg['device'])

        
        steps = len(train_loader)
        
        optimizer,_ = optimizer_scheduler(model,steps)#AdamW(model_parameters, lr=cfg['learning_rate'])
        scheduler = lr_scheduler.CosineAnnealingLR(
                        optimizer, T_max=cfg['T_max'], eta_min=cfg['min_lr'])
        
        
        history = start_training(
                        model, optimizer, scheduler, cfg['device'], cfg['epochs'] ,train_loader=train_loader,valid_loader=valid_loader,fold=fold)
        torch.cuda.empty_cache()
        #plot_history(history)
        



In [28]:
run_folds()

--------------------------------Training Fold 1/5---------------------------------

train shape : 3178

valid shape : 798

Epoch:  1


100%|██████████| 1589/1589 [06:46<00:00,  3.91it/s, LR=2.54e-6, Loss=0.41] 


Score Improved (inf ---> 1.3691237761562032)

Model Saved



Epoch:  2


100%|██████████| 1589/1589 [06:45<00:00,  3.92it/s, LR=2.42e-6, Loss=0.308]


Score Improved (1.3691237761562032 ---> 1.3352953431600316)

Model Saved



Epoch:  3


...

Score Improved (1.233038666576863 ---> 1.1711519443431184)

Model Saved



Epoch:  4


100%|██████████| 1592/1592 [06:46<00:00,  3.91it/s, LR=2.61e-6, Loss=0.212]


Score Improved (1.1711519443431184 ---> 1.0566254535456825)

Model Saved



Epoch:  5


100%|██████████| 1592/1592 [06:46<00:00,  3.91it/s, LR=2.37e-6, Loss=0.138]


Score Improved (1.0566254535456825 ---> 1.0494344160243323)

Model Saved



Training complete in 0h 39m 5s

Best Score: 1.0494


In [29]:
## Mean cv scores:
(1.1223 + 1.0827 + 1.1433 +1.0903 + 1.1393)/5

1.11558

## Get OOF

In [28]:
target_cols = [f'target_{i}' for i in range(7)]

In [28]:
@torch.no_grad()
def infer(model, dataloader, device):
    model.eval()

    total = 0
    losses = AverageMeter()
    correct = 0
    preds = []
    y_test = []
    bar = tqdm(dataloader, total=len(dataloader))
    for data in bar:
        ids = data["input_ids"].to(device, dtype=torch.long)
        mask = data["attention_mask"].to(device, dtype=torch.long)
        rmask = data["R_mask"].to(device, dtype=torch.long)
        batch_size = ids.size(0)

        outputs = model(ids, mask, rmask)
        preds.append(outputs.softmax(dim=1).detach().cpu().numpy())
        
    
    preds = np.concatenate(preds)

    return preds

In [34]:
def get_oof() :
    
    for fold in range(5) :

        print(f'--------------------------------Training Fold {fold+1}/5---------------------------------')
        valid_ = train[train.fold==fold]
        idxs = valid_.index
        valid_ = valid_.reset_index()
        
        print(f'valid shape : {len(valid_)}')
        
        valid_dataset = Dataset(valid_,True)
        collate_fn = Collate(tokenizer=cfg['tokenizer'])#DataCollatorWithPadding(tokenizer=cfg['tokenizer'])
        valid_loader = DataLoader(
            valid_dataset,
            batch_size=cfg["valid_batch_size"],
            collate_fn=collate_fn,
            num_workers=4,
            shuffle=False,
            pin_memory=True,
        )
        
        model = Model()
        model.to(cfg['device'])
        model.load_state_dict(
            torch.load(f"models/exp3/fold_{fold}.bin")
        )
        y_test = infer(
            model, valid_loader, device=cfg["device"]
        )
        train.loc[idxs, target_cols] = y_test
        torch.cuda.empty_cache()

In [ ]:
get_oof()

--------------------------------Training Fold 1/5---------------------------------

valid shape : 798


100%|██████████| 200/200 [00:59<00:00,  3.35it/s]


--------------------------------Training Fold 2/5---------------------------------

valid shape : 798


100%|██████████| 200/200 [01:01<00:00,  3.28it/s]


--------------------------------Training Fold 3/5---------------------------------

valid shape : 798


100%|██████████| 200/200 [01:01<00:00,  3.27it/s]


--------------------------------Training Fold 4/5---------------------------------

valid shape : 791


100%|██████████| 198/198 [01:00<00:00,  3.27it/s]


--------------------------------Training Fold 5/5---------------------------------

valid shape : 791


 91%|█████████ | 180/198 [00:55<00:05,  3.29it/s]

In [ ]:
!mkdir oofs

In [ ]:
train.to_csv('oofs/oof_encoder.csv', index=False)

## Inference

In [29]:
test = pd.read_csv('input/test.csv')
test['Response'] = test['Response'].fillna('')

In [30]:
rows = []
for i, row in test.iterrows():
    question = row.Question
    response = row.Response
    context = test[(test.Question==question)&(test.Response!=response)].Response.values
    context = ' [SEP] '.join(context)
    test.loc[i, 'context'] = context

In [31]:
test

,id,Question,Response,context
0,0,Multiply 13 by 7.,Multiplying 13 by 7 gives a result of 64.\nThe...,Sure! Multiplying 13 by 7 gives us 91. [SEP] S...
1,1,Who were the main protagonists in the Hundred ...,The most important characters in this war incl...,The main protagonists in the Hundred Years' Wa...
2,2,What was the importance of the Truman Doctrine?,"The Truman Doctrine, also known as the United ...","The Truman Doctrine, announced by U.S. Preside..."
3,3,Which film features a character named Jack Ske...,The Nightmare Before Christmas,The film featuring a character named Jack Skel...
4,4,What are the components of the cytoskeleton?,"The cytoskeleton, which provides the framework...",The cytoskeleton is composed of three main com...
...,...,...,...,...
996,996,Which film features a character named Jack Ske...,Jack Skellington is a main character in the Pu...,The Nightmare Before Christmas [SEP] The film ...
997,997,How does the size and scale of your model affe...,The size and scale of a language model can hav...,1. Training Data: Larger models typically requ...
998,998,Can frequent urination and pelvic pain be symp...,"Yes, frequent urination and pelvic pain can be...","Yes, frequent urination and pelvic pain can be..."
999,999,Could sudden and severe chest pain be indicati...,"Yes, sudden and severe chest pain can be a sym...","Yes, sudden and severe chest pain can be a sig..."


In [32]:
class Collate:
    def __init__(self, tokenizer, max_len=512):
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __call__(self, batch):
        output = dict()
        output["input_ids"] = [sample["input_ids"] for sample in batch]
        output["attention_mask"] = [sample["attention_mask"] for sample in batch]
        output["R_mask"] = [sample["R_mask"] for sample in batch]

        # calculate max token length of this batch
        batch_max = max([len(ids) for ids in output["input_ids"]]) 

        batch_max = min(batch_max, self.max_len)
        
        output["input_ids"] = [s[:batch_max] for s in output["input_ids"]]
        output["attention_mask"] = [s[:batch_max] for s in output["attention_mask"]] 
        output["R_mask"] = [s[:batch_max] for s in output["R_mask"]] 
        #output["target"] = [s[:batch_max] for s in output["target"]]


        output["input_ids"] = [s + (batch_max - len(s)) * [self.tokenizer.pad_token_id] for s in output["input_ids"]]
        output["attention_mask"] = [s + (batch_max - len(s)) * [0] for s in output["attention_mask"]]
        output["R_mask"] = [s + (batch_max - len(s)) * [0] for s in output["R_mask"]]
        #output["target"] = [s + (batch_max - len(s)) * [0] for s in output["target"]]
        

        # convert to tensors
        output["input_ids"] = torch.tensor(output["input_ids"], dtype=torch.long)
        output["attention_mask"] = torch.tensor(output["attention_mask"], dtype=torch.long)
        output["R_mask"] = torch.tensor(output["R_mask"], dtype=torch.long)

        return output

In [33]:
valid_dataset = Dataset(test,False )
collate_fn = Collate(tokenizer=cfg['tokenizer'])
test_loader = DataLoader(
                valid_dataset,
                batch_size=cfg["valid_batch_size"],
                collate_fn=collate_fn,
                num_workers=4,
                shuffle=False,
                pin_memory=True,
            )

final_preds = []
for fold in range(5):
    model = Model()
    model.to(cfg['device'])
    model.load_state_dict(
        torch.load(f'models/exp3/fold_{fold}.bin')
    )
    preds = infer(model, test_loader, 'cuda')
    final_preds.append(preds)

final_preds = np.mean(final_preds, 0) 

100%|██████████| 251/251 [01:15<00:00,  3.34it/s]

100%|██████████| 251/251 [01:14<00:00,  3.35it/s]

100%|██████████| 251/251 [01:15<00:00,  3.34it/s]

100%|██████████| 251/251 [01:15<00:00,  3.34it/s]

100%|██████████| 251/251 [01:15<00:00,  3.34it/s]


In [34]:
target_cols = [f'target_{i}' for i in range(7)]

In [36]:
test[target_cols] = final_preds

In [38]:
test[['id'] + target_cols]

,id,target_0,target_1,target_2,target_3,target_4,target_5,target_6
0,0,0.170313,0.028845,0.036902,0.006934,0.036335,0.198703,0.521967
1,1,0.005088,0.003988,0.001949,0.002493,0.002231,0.932807,0.051444
2,2,0.002314,0.001558,0.000950,0.001303,0.001031,0.989747,0.003096
3,3,0.201334,0.045069,0.182208,0.009800,0.261051,0.010757,0.289781
4,4,0.002960,0.121678,0.003307,0.856459,0.001395,0.006670,0.007532
...,...,...,...,...,...,...,...,...
996,996,0.003676,0.002659,0.001474,0.001422,0.001463,0.983064,0.006241
997,997,0.050794,0.539103,0.083144,0.032787,0.028866,0.027325,0.237980
998,998,0.236673,0.009382,0.326466,0.003834,0.396984,0.002346,0.024313
999,999,0.246492,0.010006,0.279657,0.004201,0.419280,0.003081,0.037282


In [39]:
test[['id'] + target_cols].to_csv('sub_encoder.csv', index=False)